<a href="https://colab.research.google.com/github/ikramkakar/Build-a-Retrieval-Augmented-Generation-RAG-/blob/main/Build_a_Retrieval_Augmented_Generation_(RAG).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y langchain langchain-community
!pip install -q langchain langchain-community openai chromadb tiktoken pypdf langchain_openai
# If you encounter ModuleNotFoundErrors for langchain components after running this cell,
# please restart the Colab runtime (Runtime -> Restart runtime) and then re-run all cells.

Found existing installation: langchain 1.2.14
Uninstalling langchain-1.2.14:
  Successfully uninstalled langchain-1.2.14
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 57.5 MB/s eta 0:00:00
   ━━━━━━

In [ ]:
import os
from google.colab import userdata

# Retrieve the API key from Colab secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Test if the key is loaded (optional)
# if os.getenv("OPENAI_API_KEY"): print("OpenAI API Key loaded successfully!")
# else: print("OpenAI API Key not found. Please set it in Colab secrets.")

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

loader = DirectoryLoader(
    "/content",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()

print(f"Loaded {len(documents)} pages from all PDFs")

Loaded 18 pages from all PDFs


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # Max number of characters in a chunk
    chunk_overlap=200,    # Overlap between chunks to maintain context
    length_function=len,
    add_start_index=True,
)

if 'documents' in locals() and documents:
    chunks = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} pages into {len(chunks)} chunks.")
    # Optional: Display the first chunk
    # if chunks: display(chunks[0].page_content)
else:
    print("No documents loaded. Please check the previous step.")
    chunks = [] # Ensure chunks is defined even if document loading failed

Split 18 pages into 32 chunks.


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

if chunks:
    embeddings = OpenAIEmbeddings()

    # Create a Chroma vector store from the document chunks and embeddings
    # This will create a local ChromaDB instance
    vectorstore = Chroma.from_documents(chunks, embeddings)
    print("Embeddings created and stored in ChromaDB.")
else:
    print("No chunks available to create embeddings. Please check chunking step.")
    vectorstore = None # Ensure vectorstore is defined

Embeddings created and stored in ChromaDB.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Retriever
retriever = vectorstore.as_retriever()

# Prompt (VERY IMPORTANT)
prompt = ChatPromptTemplate.from_template(
    "Answer the question based only on the context:\n\n{context}\n\nQuestion: {question}"
)

# RAG Chain
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Query
query = "What is the notice period for a Medical Officer who wants to resign, and can they buy out the notice period??"
response = rag_chain.invoke(query)

print(response)

The notice period for a Medical Officer who wants to resign is 60 days. Yes, they can buy out the notice period at the Basic Pay rate with prior CHRO approval.


In [ ]:
query = "What steps must a nurse follow before administering insulin to a patient?"
response = rag_chain.invoke(query)

print(response)

Before administering insulin to a patient, a nurse must follow the Five Rights of Medication Administration, which include:

1. **Right Patient**: Check the wristband ID and obtain verbal confirmation of the patient's name and date of birth.
2. **Right Medication**: Cross-check the order in the Health Information System (HIS) and verify the physical label on the insulin.
3. **Right Dose**: Recalculate the dose if it is weight-based and double-check with a second nurse for high-alert drugs like insulin.
4. **Right Route**: Confirm the route of administration as per the doctor's order and check the patient's ability to swallow if applicable.
5. **Right Time**: Administer the insulin within 30 minutes before or after the scheduled time.

Additionally, since insulin is considered a high-alert medication, it requires mandatory double-checking by two registered nurses before administration.


In [ ]:
query = "How many days of Earned Leave is a staff nurse entitled to per year, and can it be encashed?"
response = rag_chain.invoke(query)

print(response)

A staff nurse is entitled to 15 days of Earned Leave (EL) per year. Earned Leave can be encashed at the time of retirement, resignation, or death of the employee, subject to a maximum of 300 days accumulated EL.


In [ ]:
query = "What PPE is required when caring for a patient under airborne isolation precautions, and in what order should it be removed?"
response = rag_chain.invoke(query)

print(response)

When caring for a patient under airborne isolation precautions, the required PPE includes:

- Disposable gown
- N95 respirator (fit-tested)
- Goggles/shield

The order for removing the PPE (doffing) is as follows:

1. Gloves (peel off, discard)
2. Hand hygiene
3. Gown (untie, roll inward, discard)
4. Hand hygiene
5. Goggles/face shield (clean outer surface)
6. Mask (remove from behind, do not touch front)
7. Hand hygiene


In [ ]:
query = "What is the notice period for a Medical Officer who wants to resign, and can they buy out the notice period?"
response = rag_chain.invoke(query)

print(response)

The notice period for a Medical Officer who wants to resign is 60 days. Yes, they can buy out the notice period at the Basic Pay rate with prior CHRO approval.
